# NB3 — Scouting Output (Final Shortlist)

Turns the scored pool into a defensible board of acquisition targets. A transparent funnel,
not a blended super-score. Each stage is a step a real recruitment department would take.

Framing: a **2022-23 scouting snapshot** (the most recent complete free advanced data after
the FBref/Opta shutdown). Mourinho's confirmed 2026-27 arrival is the motivation for the
archetype. The age cap keeps targets realistic for that horizon.

Funnel: exclude current RM squad → age ≤ 28 (latest-season age) → focus the gap roles NB2
flagged (striker, defensive mid, full-back) → rank by Mourinho fit → layer value → take 2-3 per area.

Inputs: `scored_pool.csv`, `pool_enriched.csv`, `madrid_squad_profile.csv`, `mourinho_weights.json`,
`tier_C_players.csv` (+ `tier_A` for the youth section's peer pool). **Run after `scoring.py`.**

In [ ]:
import sys, json, unicodedata
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TRANS = ROOT/"data"/"transformed"
OUTD  = ROOT/"data"/"outputs"; OUTD.mkdir(parents=True, exist_ok=True)
RADAR = ROOT/"assets"/"radar_charts"; RADAR.mkdir(parents=True, exist_ok=True)
sys.path.append(str(ROOT/"src"))
import scoring  # our engine

AGE_CAP    = 26 # Max player age in 2026-27 would be 30
GAP_ROLES  = ["striker", "defensive_mid", "full_back"]   # NB2-flagged thin/ageing areas
N_PER_AREA = 3
# why each role is a target (designated from NB2's squad read) — used in rationales
GAP_REASON = {
    "striker":       "thin (only 2 senior strikers)",
    "defensive_mid": "thin & ageing (one senior holding mid)",
    "full_back":     "ageing starters (Carvajal 33, Mendy 30)",
}
# value bands in EUR millions (peak value)
def value_band(m):
    if pd.isna(m): return "unknown"
    return "marquee" if m >= 80 else "mid" if m >= 30 else "bargain"

weights = json.load(open(ROOT/"config"/"mourinho_weights.json", encoding="utf-8"))
def norm(s): return " ".join(unicodedata.normalize("NFKD",str(s)).encode("ascii","ignore").decode().lower().split())
def pretty(k): return k.replace("_"," ").replace("plus","+").title()

## Step 1 — Load scored pool, latest-season age, current squad

In [9]:
scored = pd.read_csv(TRANS/"scored_pool.csv")
pool   = pd.read_csv(TRANS/"pool_enriched.csv")
squad  = pd.read_csv(TRANS/"madrid_squad_profile.csv")

# age + value as of the player's latest season (value lives in pool_enriched, not scored_pool)
meta = (pool.sort_values("Season_End_Year").groupby("Url").tail(1)
            [["Url","Age","market_value_eur"]].rename(columns={"Age":"age_snap"}))
scored = scored.merge(meta, on="Url", how="left")
scored["valM"]  = (scored["market_value_eur"]/1e6).round(1)
scored["band"]  = scored["valM"].map(value_band)

squad_names = set(squad["name"].map(norm))
squad_depth = squad["role"].value_counts().to_dict()
print(f"pool {len(scored)} | current RM squad {len(squad_names)} | gap roles {GAP_ROLES}")

pool 641 | current RM squad 36 | gap roles ['striker', 'defensive_mid', 'full_back']


## Step 2 — The funnel (exclude squad → age ≤ 28 → gap roles)

In [10]:
scored["_k"] = scored["Player"].map(norm)
already = scored["_k"].isin(squad_names)
elig = scored[~already].copy()
print(f"excluded {already.sum()} players already at Real Madrid (e.g. Mbappe, Tchouameni, Vinicius)")

elig = elig[elig["age_snap"] <= AGE_CAP]
print(f"after age<= {AGE_CAP}: {len(elig)} players remain")

board = elig[elig["role"].isin(GAP_ROLES)].copy()
print(f"gap-role candidates: {len(board)}")
# full ranked board per gap role (for the app) -> save
board_sorted = board.sort_values(["role","mourinho_score"], ascending=[True,False])
board_sorted.to_csv(OUTD/"shortlist_board_full.csv", index=False)

excluded 8 players already at Real Madrid (e.g. Mbappe, Tchouameni, Vinicius)
after age<= 26: 247 players remain
gap-role candidates: 95


## Step 3 — Pick 2-3 per area, flag the lead target

In [11]:
BANDS = ["marquee", "mid", "bargain"]   # value tiers (thresholds set in cell 1)

picks = []
for b in BANDS:
    for r in GAP_ROLES:
        sub = (board[(board["band"] == b) & (board["role"] == r)]
               .sort_values("mourinho_score", ascending=False).head(1).copy())
        if sub.empty:
            print(f"  (no candidate: {b:8s} / {r})")
            continue
        picks.append(sub)

shortlist = pd.concat(picks, ignore_index=True)
# lead target = the single best-fit option in each area, regardless of price band
shortlist["lead_target"] = (shortlist.groupby("role")["mourinho_score"]
                                     .transform("max").eq(shortlist["mourinho_score"]))

cols = ["band", "role", "Player", "Squad", "age_snap", "valM", "mourinho_score", "lead_target"]
shortlist[cols].to_csv(OUTD / "shortlist_final.csv", index=False)
print(shortlist[cols].to_string(index=False))

   band          role            Player           Squad  age_snap  valM  mourinho_score  lead_target
marquee       striker    Erling Haaland        Dortmund      22.0 200.0       83.727273         True
marquee defensive_mid             Rodri Manchester City      26.0 130.0       60.193548        False
marquee     full_back   Alphonso Davies   Bayern Munich      22.0  80.0       79.626087         True
    mid       striker       André Silva  Eint Frankfurt      26.0  45.0       74.571429        False
    mid defensive_mid     Wilfred Ndidi  Leicester City      25.0  60.0       78.516129         True
    mid     full_back Aaron Wan-Bissaka  Manchester Utd      24.0  40.0       67.339130        False
bargain       striker       Terem Moffi         Lorient      23.0  25.0       45.402597        False
bargain defensive_mid Diadie Samassékou      Hoffenheim      26.0  20.0       65.387097        False
bargain     full_back   Fabien Centonze            Metz      26.0   8.0       68.695652    

## Step 4 — Annotate: strengths, weaknesses, one-line rationale

Strengths/weaknesses are read straight from each player's percentile fingerprint (`pct__` columns),
so the annotation is auditable, not asserted.

In [12]:
def player_pcts(row, role):
    out = {}
    for mname in weights["roles"][role]["metrics"]:
        c = f"pct__{mname}"
        if c in row and pd.notna(row[c]):
            out[pretty(mname)] = float(row[c])
    return out

rows = []
for _, p in shortlist.iterrows():
    pc = player_pcts(p, p["role"])
    ordered = sorted(pc.items(), key=lambda x: x[1], reverse=True)
    strengths = ", ".join(f"{k} ({v:.0f})" for k,v in ordered[:3])
    weaks     = ", ".join(f"{k} ({v:.0f})" for k,v in ordered[-2:])
    rationale = (f"{p['Player']} ({int(p['age_snap'])}, {p['Squad']}) — {p['band']} {p['role']} "
                 f"fit {p['mourinho_score']:.0f}. Gap: {GAP_REASON[p['role']]}. "
                 f"Top traits: {strengths}.")
    rows.append({"Player":p["Player"], "role":p["role"], "lead_target":p["lead_target"],
                 "valM":p["valM"], "band":p["band"], "fit":round(p["mourinho_score"],1),
                 "strengths":strengths, "weaknesses":weaks, "rationale":rationale})
annot = pd.DataFrame(rows)
annot.to_csv(OUTD/"shortlist_annotated.csv", index=False)
for r in rows: print("•", r["rationale"], "\n")

• Erling Haaland (22, Dortmund) — marquee striker fit 84. Gap: thin (only 2 senior strikers). Top traits: Non Pen Xg (99), Non Pen Goals (99), Goal Creating Actions (97). 

• Rodri (26, Manchester City) — marquee defensive_mid fit 60. Gap: thin & ageing (one senior holding mid). Top traits: Recoveries (100), Aerial Win Pct (100), Pass Completion (98). 

• Alphonso Davies (22, Bayern Munich) — marquee full_back fit 80. Gap: ageing starters (Carvajal 33, Mendy 30). Top traits: Progressive Carries (100), Recoveries (100), Pass Completion (94). 

• André Silva (26, Eint Frankfurt) — mid striker fit 75. Gap: thin (only 2 senior strikers). Top traits: Prog Passes Received (95), Non Pen Xg (90), Touches In Box (88). 

• Wilfred Ndidi (25, Leicester City) — mid defensive_mid fit 79. Gap: thin & ageing (one senior holding mid). Top traits: Tackles + Int (100), Recoveries (98), Interceptions (97). 

• Aaron Wan-Bissaka (24, Manchester Utd) — mid full_back fit 67. Gap: ageing starters (Carvajal 3

## Step 5 — Radar per shortlisted player (percentile vs role)

In [13]:
def radar_png(name, role, pcts, path):
    labels=list(pcts.keys()); vals=list(pcts.values()); N=len(labels)
    ang=np.linspace(0,2*np.pi,N,endpoint=False).tolist(); ang+=ang[:1]; v=vals+vals[:1]
    fig,ax=plt.subplots(figsize=(6,6),subplot_kw=dict(polar=True))
    ax.plot(ang,v,color="#c8102e",lw=2); ax.fill(ang,v,color="#c8102e",alpha=0.25)
    ax.set_xticks(ang[:-1]); ax.set_xticklabels(labels,fontsize=7)
    ax.set_ylim(0,100); ax.set_yticks([25,50,75]); ax.set_yticklabels(["25","50","75"],fontsize=6)
    ax.set_title(f"{name} — {role}\n(percentile vs role, 2022-23)",fontsize=11,pad=18)
    fig.savefig(path,dpi=120,bbox_inches="tight"); plt.close(fig)

for f in RADAR.glob("*.png"): f.unlink()   # clear stale radars before regenerating

made=[]
for _, p in shortlist.iterrows():
    pc = player_pcts(p, p["role"])
    fn = RADAR/(norm(p["Player"]).replace(" ","_")+".png")
    radar_png(p["Player"], p["role"], pc, fn); made.append(fn.name)
print(f"{len(made)} radars saved to assets/radar_charts/:"); print(made)

9 radars saved to assets/radar_charts/:
['erling_haaland.png', 'rodri.png', 'alphonso_davies.png', 'andre_silva.png', 'wilfred_ndidi.png', 'aaron_wan-bissaka.png', 'terem_moffi.png', 'diadie_samassekou.png', 'fabien_centonze.png']


## Step 6 — Youth watchlist (bonus, future value)

Tier C prospects scored *against the senior pool* (not just other kids), so the percentile shows
how a teenager already ranks among established players. No file changes. Just score a
Tier A + Tier C union and keep the Tier C rows.

In [14]:
tA = pd.read_csv(TRANS/"tier_A_players.csv"); tC = pd.read_csv(TRANS/"tier_C_players.csv")
union = pd.concat([tA, tC], ignore_index=True)
agg = scoring.aggregate_player_seasons(union, weights)
sc  = scoring.score_pool(agg, weights)
cset = set(tC["Url"])
youth = sc[sc["Url"].isin(cset)].copy()
youth = youth.merge(union[["Url","market_value_eur"]].drop_duplicates("Url"), on="Url", how="left")
youth["valM"] = (youth["market_value_eur"]/1e6).round(1)
yv = youth[youth["role"].isin(GAP_ROLES)].sort_values("mourinho_score", ascending=False)
keep = ["Player","Squad","role","Age","valM","mourinho_score"]
yv[keep].to_csv(OUTD/"youth_watchlist.csv", index=False)
print("Top young prospects in gap roles (scored vs senior pool):")
print(yv[keep].head(10).to_string(index=False))

Top young prospects in gap roles (scored vs senior pool):
           Player          Squad          role  Age  valM  mourinho_score
       Malo Gusto           Lyon     full_back 18.0  35.0       79.392000
      Melvin Bard           Nice     full_back 20.0  15.0       73.040000
    Lucien Agoume          Brest defensive_mid 19.0   7.0       68.323529
        Vanderson         Monaco     full_back 20.0  20.0       63.296000
   Fabiano Parisi         Empoli     full_back 20.0  10.0       62.216000
      Nuno Mendes      Paris S-G     full_back 19.0  75.0       60.912000
 Georginio Rutter     Hoffenheim       striker 19.0  32.0       59.300000
   Quentin Merlin         Nantes     full_back 19.0  15.0       52.552000
      Luke Thomas Leicester City     full_back 20.0  12.0       45.440000
Arnaud Kalimuendo           Lens       striker 19.0  30.0       45.300000


## Step 7 — Caveats baked into the deliverable

1. 2022-23 form basis — a pick is "was an excellent fit then"; the ≤28 cap keeps it realistic for 2026-27.
2. Defender volume inflation — low-block teams (Metz, Atalanta) pad tackles/clearances; read raw defensive
   volume with that in mind. A possession-adjusted v2 would refine this.
3. One reading of Mourinho — the weights are an authored prior, transparent and editable, not objective truth.

Outputs (`data/outputs/`): `shortlist_final.csv`, `shortlist_annotated.csv`, `shortlist_board_full.csv`,
`youth_watchlist.csv`; radars in `assets/radar_charts/`. These feed the Streamlit app's shortlist page.